# Sesión 5 · Notebook previo
## Un conjunto de evaluación completo y explicabilidad con SHAP

**Curso:** Ciencia de datos aumentada con IA: del dato a la decisión
**Tiempo estimado:** 45 a 55 minutos · **Entrega:** este notebook ejecutado, antes de la clase

### Qué vas a hacer
1. Consolidar en un solo lugar las evaluaciones que han hecho desde la sesión 1 (extracción, RAG, agente) en un **conjunto de evaluación** con tres niveles: pruebas automáticas, juicio de un LLM y revisión humana.
2. Usar un **LLM como juez** para calificar respuestas abiertas, y medir qué tan de acuerdo está con una persona.
3. Calcular **SHAP** sobre un modelo tabular para explicar sus predicciones caso por caso.
4. Dejar una checklist lista para el taller, donde auditarán sesgos, seguridad y costo del prototipo completo.

Este notebook retoma la idea de Hamel Husain (Your AI product needs evals): los productos con IA generativa que funcionan bien casi siempre tienen, detrás, un sistema de evaluación serio.

## 0. Configuración

In [ ]:

import os, json, re
import numpy as np
import pandas as pd
from google import genai
from getpass import getpass

def obtener_clave(nombre):
    try:
        from google.colab import userdata
        clave = userdata.get(nombre)
        if clave:
            return clave
    except Exception:
        pass

    return os.environ.get(nombre) or getpass(f"Pega tu {nombre}: ")

cliente = genai.Client(api_key=obtener_clave("GEMINI_API_KEY"))
MODELO = "gemini-3.5-flash-lite"


## 1. Los tres niveles de evaluación
Siguiendo a Husain, un sistema de evaluación completo tiene tres niveles, de más barato/rápido a más costoso/lento:

1. **Pruebas automáticas (unit tests):** reglas simples y baratas, como "¿el JSON es válido?" o "¿el monto extraído es un número positivo?". Ya las usaste en la sesión 1.
2. **Evaluación humana y con LLM como juez:** para respuestas abiertas (como las del RAG o el agente), donde no hay una única respuesta "correcta" exacta.
3. **Pruebas A/B con usuarios reales:** para productos ya en producción con tráfico real (fuera del alcance de este curso, pero se menciona en el taller).

Vamos a construir el nivel 2: un LLM como juez, calibrado contra el criterio humano.

## 2. LLM como juez
La idea es simple pero delicada: usar un LLM para calificar la respuesta de **otro** LLM (o del mismo, en otra llamada), con un criterio explícito. El riesgo es que el juez tenga sus propios sesgos, así que **siempre se valida contra una muestra calificada por una persona**, igual que hicimos con la extracción de datos en la sesión 1.

In [ ]:
# Reutilizamos un conjunto de respuestas de ejemplo del RAG de la sesión 2 (pregunta, respuesta del RAG, y una
# calificación humana de referencia: 1 = mala, 2 = aceptable, 3 = buena).
CASOS_JUEZ = [
    ("¿Cuánto tiempo tengo para devolver un producto sin dar motivo?",
     "Tienes 5 días hábiles desde que recibes el producto, siempre que esté sin usar y en su empaque original. (Fuente: politica_devoluciones_garantias.md)", 3),
    ("¿Cuánto tiempo tengo para devolver un producto sin dar motivo?",
     "Puedes devolver el producto cuando quieras, TiendaTech siempre lo acepta.", 1),
    ("¿Cuál es la garantía de la batería del portátil Lenovo IdeaPad?",
     "La batería tiene 24 meses de garantía si el deterioro es un defecto de fabricación. (Fuente: manual_portatil_lenovo_ideapad.md)", 3),
    ("¿Cuál es la garantía de la batería del portátil Lenovo IdeaPad?",
     "No tengo información sobre eso.", 1),
    ("¿Cuánto cuesta enviar un pedido de $50.000?",
     "El envío para compras menores a $200.000 cuesta entre $12.000 y $35.000 COP según la ciudad. (Fuente: politica_envios.md)", 3),
    ("¿Cuánto cuesta enviar un pedido de $50.000?",
     "El envío siempre es gratis en TiendaTech.", 1),
    ("¿Puedo cambiar la dirección de mi pedido después de comprarlo?",
     "Sí, siempre puedes cambiarla sin restricciones.", 1),
    ("¿Puedo cambiar la dirección de mi pedido después de comprarlo?",
     "Solo si el pedido no ha salido del centro de distribución; una vez despachado no se puede. (Fuente: politica_envios.md)", 3),
]
print(f"{len(CASOS_JUEZ)} casos con calificación humana de referencia")

In [ ]:
import time

# Redefinimos el modelo a uno que esté activo y disponible
MODELO = "gemini-3.5-flash-lite"

PROMPT_JUEZ = """Eres un evaluador estricto de un sistema de atención al cliente basado en RAG.
Vas a calificar la RESPUESTA a la PREGUNTA según esta escala:
1 = mala (inventa información, contradice la política, o no responde cuando sí podía)
2 = aceptable (responde algo correcto pero incompleto, o le falta citar la fuente)
3 = buena (responde correctamente, con base en la política, y cita la fuente cuando corresponde)

Responde ÚNICAMENTE con el número (1, 2 o 3), sin explicación.

PREGUNTA: {pregunta}
RESPUESTA: {respuesta}
CALIFICACIÓN:"""

def calificar_con_juez(pregunta, respuesta):
    r = cliente.models.generate_content(
        model=MODELO,
        contents=PROMPT_JUEZ.format(pregunta=pregunta, respuesta=respuesta)
    )
    texto = r.text.strip()
    m = re.search(r"[123]", texto)
    return int(m.group()) if m else None

filas = []
for i, (pregunta, respuesta, calificacion_humana) in enumerate(CASOS_JUEZ):
    # Añadimos una pausa entre llamadas para evitar el error de cuota (429 RESOURCE_EXHAUSTED)
    if i > 0:
        print(f"Esperando 15 segundos antes de calificar el caso {i+1}/{len(CASOS_JUEZ)} para respetar la cuota...")
        time.sleep(15)

    calificacion_juez = calificar_con_juez(pregunta, respuesta)
    filas.append({"pregunta": pregunta[:40] + "...", "calificacion_humana": calificacion_humana,
                  "calificacion_juez": calificacion_juez, "coincide": calificacion_humana == calificacion_juez})

df_juez = pd.DataFrame(filas)
df_juez

In [ ]:
acuerdo = df_juez["coincide"].mean()
print(f"Acuerdo entre el juez LLM y la calificación humana: {acuerdo:.0%}")

### TU TURNO 1
1. ¿En qué casos el juez no coincidió con la calificación humana? ¿Se equivocó "por poco" (1 vs 2) o "en grande" (1 vs 3)?
2. Ajusta el `PROMPT_JUEZ` (por ejemplo, con un ejemplo de cada categoría, few-shot) y vuelve a correr la evaluación. ¿Mejoró el acuerdo?

In [5]:
# TU TURNO: ajusta PROMPT_JUEZ arriba y vuelve a ejecutar la celda de calificación

respuesta_1 = """
Escribe aquí tu análisis.
"""

## 3. Consolidando el conjunto de evaluación del proyecto
Cada equipo debería tener, para su prototipo, una tabla como esta que junte los tres tipos de prueba. Complétala con los datos de tu propio proyecto (puedes usar los de las sesiones 1 a 3 como ejemplo si aún no tienes los tuyos).

In [ ]:
CONJUNTO_EVALUACION = pd.DataFrame([
    {"componente": "Extracción a JSON (sesión 1)", "tipo_prueba": "Automática", "metrica": "Exactitud por campo", "resultado": None, "n_casos": 10},
    {"componente": "RAG (sesión 2)", "tipo_prueba": "Automática", "metrica": "Hit rate@3 / MRR", "resultado": None, "n_casos": 10},
    {"componente": "RAG (sesión 2)", "tipo_prueba": "LLM como juez + humana", "metrica": "Calificación 1-3, % acuerdo", "resultado": None, "n_casos": 8},
    {"componente": "Agente (sesión 3)", "tipo_prueba": "Automática + humana", "metrica": "% de preguntas correctas", "resultado": None, "n_casos": 10},
    {"componente": "Modelo tabular / series (sesión 4)", "tipo_prueba": "Automática", "metrica": "Accuracy / F1 / MAE", "resultado": None, "n_casos": None},
])
CONJUNTO_EVALUACION  # TU TURNO: completa la columna 'resultado' con tus propios números

## 4. Explicabilidad con SHAP
SHAP asigna, para cada predicción individual, cuánto aportó cada variable a alejar la predicción del promedio. Es distinto de la importancia de variables global: aquí vemos el **por qué de un caso específico**, que es lo que se necesita para auditar o explicarle una decisión a una persona.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
import shap

tickets = pd.read_csv("tickets_soporte.csv").dropna(subset=["satisfaccion"]).copy()
tickets["satisfaccion_baja"] = (tickets["satisfaccion"] <= 2).astype(int)

caracteristicas = ["canal", "categoria", "horas_resolucion"]
X = tickets[caracteristicas].copy()
codificadores = {}
for col in ["canal", "categoria"]:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    codificadores[col] = le
y = tickets["satisfaccion_baja"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1, stratify=y)
modelo = GradientBoostingClassifier(random_state=1).fit(X_train, y_train)

explicador = shap.TreeExplainer(modelo)
valores_shap = explicador.shap_values(X_test)
print("Forma de los valores SHAP:", np.array(valores_shap).shape)

In [ ]:
# Importancia global: qué variable pesa más en promedio
importancia_global = pd.Series(np.abs(valores_shap).mean(axis=0), index=caracteristicas).sort_values(ascending=False)
print("Importancia global (promedio de |SHAP|):")
print(importancia_global)

In [ ]:
# Explicación de un caso puntual: el primer ticket del conjunto de prueba
i = 0
caso = X_test.iloc[i]
print("Caso:", caso.to_dict())
print("Predicción del modelo (satisfaccion_baja):", modelo.predict(X_test.iloc[[i]])[0])
print("\nAporte de cada variable a esta predicción (SHAP):")
for var, valor_shap in zip(caracteristicas, valores_shap[i]):
    signo = "aumenta" if valor_shap > 0 else "reduce"
    print(f"  {var} = {caso[var]}: {signo} la probabilidad de satisfacción baja (SHAP={valor_shap:.3f})")

### TU TURNO 2
1. Elige otro caso del conjunto de prueba (cambia `i`) donde el modelo prediga satisfacción baja. ¿Qué variable domina la explicación?
2. Si tuvieras que explicarle a un gerente, en una frase, por qué el modelo marcó ese ticket como riesgo de baja satisfacción, ¿qué le dirías?
3. ¿Ves algo en `importancia_global` que te preocupe desde el punto de vista de sesgo (por ejemplo, que el canal pese demasiado)?

In [11]:
respuesta_2 = """
Escribe aquí tus respuestas.
"""

## 5. Cierre
### TU TURNO 3 · Reflexión final (máx. 5 líneas)
¿Qué parte de tu prototipo (del proyecto) es más urgente evaluar con el nivel 2 (LLM como juez + humano) antes de la sustentación, y por qué esa y no otra?

In [12]:
respuesta_3 = """
Escribe aquí tu reflexión.
"""

DECLARACION_IAG = """
Nivel de uso de IAG: (1 a 5, según el acuerdo del curso)
Herramientas usadas y para qué:
Cómo verifiqué la calidad y veracidad del contenido generado:
"""

### Antes de entregar
- Sube `tickets_soporte.csv` a tu sesión de Colab antes de ejecutar.
- Ejecuta **Entorno de ejecución → Reiniciar y ejecutar todo** y verifica que no haya errores.
- Comprueba que las celdas `respuesta_1` a `respuesta_3`, `CONJUNTO_EVALUACION` y `DECLARACION_IAG` estén completas.
- **No compartas tu clave de API.**